In [29]:
import pandas as pd
import os
import json
import numpy as np

In [36]:
# ------------------------------
# CONFIGURATION
# ------------------------------
DATA_DIR = "/content/"  # Folder containing incidents.csv and classification CSVs
INCIDENT_FILE = os.path.join(DATA_DIR, "incidents.csv")
CLASSIFICATION_FILES = [
    "classifications_CSETv0.csv",
    "classifications_CSETv1.csv",
    "classifications_CSETv1_Annotator-1.csv",
    "classifications_CSETv1_Annotator-2.csv",
    "classifications_CSETv1_Annotator-3.csv",
    "classifications_MIT.csv",
    "classifications_GMF.csv"
]

OUTPUT_JSON = os.path.join(DATA_DIR, "unified_incidents.json")

In [31]:
# ------------------------------
# HELPER FUNCTIONS
# ------------------------------
def load_csv(file_path):
    if os.path.exists(file_path):
        return pd.read_csv(file_path)
    else:
        print(f"[WARN] File not found: {file_path}")
        return pd.DataFrame()

def clean_value(val):
    """Convert NaN to None, fix boolean/int issues, parse stringified lists"""
    if pd.isna(val):
        return None
    if isinstance(val, (np.bool_, bool)):
        return bool(val)
    if isinstance(val, (np.integer, int, np.floating, float)):
        return val.item() if hasattr(val, 'item') else val
    # Convert stringified list to actual list if possible
    if isinstance(val, str) and val.startswith("[") and val.endswith("]"):
        try:
            return json.loads(val.replace("'", '"'))
        except:
            return val
    return val

def clean_dict(d):
    """Recursively clean a dictionary"""
    return {k: clean_value(v) if not isinstance(v, dict) else clean_dict(v) for k, v in d.items()}


In [32]:
# ------------------------------
# LOAD INCIDENTS
# ------------------------------
incidents_df = load_csv(INCIDENT_FILE)
incidents_df["incident_id"] = incidents_df["incident_id"].astype(str)

unified_incidents = {}
for _, row in incidents_df.iterrows():
    incident_id = str(row["incident_id"])
    unified_incidents[incident_id] = {
        "incident_id": incident_id,
        "title": row.get("title"),
        "description": row.get("description"),
        "date": row.get("date"),
        "entities": clean_value(row.get("Alleged harmed or nearly harmed parties")),
        "tags": clean_value(row.get("Alleged deployer of AI system")),
        "classifications": {}
    }


In [33]:
# ------------------------------
# MERGE CLASSIFICATIONS
# ------------------------------
for file_name in CLASSIFICATION_FILES:
    full_path = os.path.join(DATA_DIR, file_name)
    df = load_csv(full_path)
    if df.empty:
        continue

    if "Incident ID" in df.columns:
        df["incident_id"] = df["Incident ID"].astype(str)
    elif "incident_id" in df.columns:
        df["incident_id"] = df["incident_id"].astype(str)
    else:
        continue

    classification_key = os.path.splitext(file_name)[0]

    for _, row in df.iterrows():
        incident_id = str(row["incident_id"])
        if incident_id not in unified_incidents:
            unified_incidents[incident_id] = {
                "incident_id": incident_id,
                "title": None,
                "description": None,
                "date": None,
                "entities": None,
                "tags": None,
                "classifications": {}
            }
        # Clean each row before adding
        row_dict = row.drop("incident_id").to_dict()
        unified_incidents[incident_id]["classifications"][classification_key] = clean_dict(row_dict)


In [38]:
# ------------------------------
# EXPORT TO JSON (SORTED)
# ------------------------------
# Sort incidents by incident_id numerically
sorted_incidents = sorted(
    unified_incidents.values(),
    key=lambda x: int(x.get("incident_id", 0))
)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(sorted_incidents, f, ensure_ascii=False, indent=2)

print(f"[INFO] Unified incidents JSON written and sorted by incident_id to {OUTPUT_JSON}")


[INFO] Unified incidents JSON written and sorted by incident_id to /content/unified_incidents.json
